# First calls and output-to-input chaining: practice

Import helpers → load configuration → construct a client → send messages → extract answer text → feed one answer into another request.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [1_foundations/1_lab1.ipynb](../../../../../1_foundations/1_lab1.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 3. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 4. Chaining: one output becomes the next input

A chain has a fixed order. Each stage depends on the previous result. Here the replies are explicit fictional fixtures, so you can observe the wiring without an API.

**Predict:** Does the second stage receive the first result, or just the original question?

In [ ]:
def choose_area():
    return "Appointment scheduling"
def describe_problem(area):
    return {"area": area, "problem": "People miss reminders."}
def propose_solution(problem):
    return f"Prototype for {problem['area']}: draft reminder messages."
area = choose_area()
problem = describe_problem(area)  # Previous output becomes this input.
solution = propose_solution(problem)
print(area, problem, solution, sep="\n")
assert area in solution

**Check:** The second stage receives the first output. Calling independent prompts without passing the result is not this chain.

**Change one thing:** Change choose_area to a different fictional business area. Trace the changed value through each stage.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.